# Initial Project Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted successfully!")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, time

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score


pd.set_option("display.width", 120)
print("Libraries loaded.")

Mounted at /content/drive
Drive mounted successfully!
Libraries loaded.


# Load and clean the raw data

Initially working with this dataset it's cleaned to turn the raw csv data into an actual clean table.

First, the stray index column, the unnamed column, is dropped as it doesn't contain real data. Afterwhich the vitals are forced to become real numbers, instead of text. As a result text such as "120 bpm" becomes NaN. Then the ESI labels must be a number ranging from 1 - 5. The rows in which that data is missing or out of range is unsuited to train a triage model; that data is dropped. Then, physically impossible vital signs are blanked out such that it doesn't mislead the model. The gender column now operates like a flag, using numbers like 0 and 1 to indicate male and female. Other missing numbers in the columns are filled wiuth the column's median number. 

In [2]:
import os 
RAW = "/content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv"
df_raw = pd.read_csv(RAW)
print("Loaded", df_raw.shape[0], "rows and", df_raw.shape[1], "columns from:", RAW)
df_raw.head()

Loaded 55121 rows and 226 columns from: /content/drive/MyDrive/yaleemmlc_admissionprediction_triage.csv


,Unnamed: 0,dep_name,esi,age,gender,ethnicity,race,lang,religion,maritalstatus,...,cc_vaginaldischarge,cc_vaginalpain,cc_weakness,cc_wheezing,cc_withdrawal-alcohol,cc_woundcheck,cc_woundinfection,cc_woundre-evaluation,cc_wristinjury,cc_wristpain
0,7,A,4.0,87.0,Female,Hispanic or Latino,Other,Other,Pentecostal,Widowed,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,17,B,2.0,53.0,Male,Hispanic or Latino,Other,English,Catholic,Significant Other,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,40,A,2.0,49.0,Female,Non-Hispanic,White or Caucasian,English,Catholic,Married,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,47,A,3.0,22.0,Female,Hispanic or Latino,White or Caucasian,English,Catholic,Single,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,60,A,2.0,62.0,Male,Non-Hispanic,White or Caucasian,English,Protestant,Divorced,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]
df = df_raw.copy()

df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

for col in VITALS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["esi"] = pd.to_numeric(df["esi"], errors="coerce")
df = df[df["esi"].isin([1, 2, 3, 4, 5])].copy()

df.loc[(df["triage_vital_temp"] < 90) | (df["triage_vital_temp"] > 110), "triage_vital_temp"] = np.nan
df.loc[df["triage_vital_o2"] > 100, "triage_vital_o2"] = np.nan

df["gender"] = df["gender"].astype(str).str.strip().str.lower().map(
    {"male": 0, "m": 0, "female": 1, "f": 1})

for col in VITALS + ["age", "gender"]:
    df[col] = df[col].fillna(df[col].median())

df["esi"] = df["esi"].astype(int)
print("Modelling table ready:", df.shape)
df["esi"].value_counts().sort_index()

Modelling table ready: (55121, 225)


,count
esi,
1,77
2,17924
3,27010
4,8896
5,1214


# Choose the features (X) and targets (y)

Here, we group each column by meaning. Vitals and CC flags here are primarily grouped. Columns like **leakage**, **admin/arriva**l and **demographics** are excluded. 

In [4]:
TARGET = "esi"

VITALS = ["triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
          "triage_vital_o2", "triage_vital_temp", "triage_glucose"]

DEMOGRAPHICS = ["age", "gender", "ethnicity", "race", "lang", "religion",
                "maritalstatus", "employstatus", "insurance_status"]

ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]

LEAKAGE = ["disposition", "previousdispo"]

FEATURES = [c for c in df.columns if c != TARGET and c not in LEAKAGE + ADMIN + DEMOGRAPHICS]

X = df[FEATURES]
y = df[TARGET]
print("Model will use", len(FEATURES), "features. First few:", FEATURES[:6])

Model will use 208 features. First few: ['triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2', 'triage_vital_o2_device']


In [5]:
#This is the exact Week 6 split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("train:", X_train.shape[0], "| test:", X_test.shape[0])

train: 44096 | test: 11025


# Discussion Point

Is *`cc_other`* worth keeping?

Since *`cc_other`* catches everything that the other flags didn't, it constitutes as a rare case that isn't usually considered. In my opinion, I'd say that the other flag be kept and investigated, rather than dismissed outright. After investigation, a decision can be made on whether the contents of the column are constructive and not just fluff.


In [6]:
if "cc_other" in df.columns:
    cc_cols = [c for c in df.columns if c.startswith("cc_")]
    total = len(df)
    has_other = int(df["cc_other"].sum())
    only_other = int(((df["cc_other"] == 1) & (df[cc_cols].sum(axis=1) == 1)).sum())
    print(f"Patients flagged cc_other: {has_other} of {total} ({has_other/total:.1%})")
    print(f"...and of those, patients whose ONLY complaint is 'other': {only_other}")
    print("\nMean ESI by cc_other flag (does 'other' lean urgent or not?):")
    print(df.groupby("cc_other")["esi"].mean().round(2))
else:
    print("No cc_other column in this sample — skipping. (On the full extract it will run.)")

Patients flagged cc_other: 4491 of 55121 (8.1%)
...and of those, patients whose ONLY complaint is 'other': 3352

Mean ESI by cc_other flag (does 'other' lean urgent or not?):
cc_other
0.0    2.87
1.0    3.01
2.0    3.26
3.0    3.00
Name: esi, dtype: float64


# Beating the Week 6 baseline

Here, the logistic regression baseline is retrained and the macro f1 is measured across all ESI levels. In this case, ESI 1 is treated equally as every other ESI level. From this point, every model has to beat this number to justify its extra cost. 

In [7]:
baseline = make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=1000, random_state=42))
baseline.fit(X_train, y_train)
baseline_f1 = f1_score(y_test, baseline.predict(X_test), average="macro")
print("Baseline (logistic regression) macro-F1:", round(baseline_f1, 3))

Baseline (logistic regression) macro-F1: 0.492


# Optimization

Generally, a model uses default settings that were chosen to be reasonable on average. This means that it isn't tuned to our data specifically; its unoptimized. Optimization means two things, feature engineering and hyperparameter tuning. 

**Feature engineering** is transforming raw data into meaningful, machine readable inputs. This gives the model better clues and info to go off of. **Hyperparameter tuning** is the process of selecting the optimal configuration of external settings. So, adjusting the model's own dials and settings. 

Both can lift performance without collecting a single new patient. 

# Feature Engineering

As mentioned before, a feature is a clue that the model uses. Feature engineering build new clues from existing columns using clinical knowledge. Two patterns, ratios/combinations and red flags are very handy in providing clues. We build them per row so that they're able to apply and train safely while also being able to be tests sparately, without leakage. 

NOTE: Blood pressure is a vital sign that is often mismeasured at triage. Other vitals, **respiratory rate**, **oxygen saturation**, **temperature** and **glucose**, can be used as features.

The code below builds new clinical features from existing vitals then applies them to the train and test splits in the same way.

In [8]:
def add_clinical_features(data):
    out = data.copy()

    # --- ratios & combinations (given as examples) ---
    out["shock_index"]    = out["triage_vital_hr"] / out["triage_vital_sbp"]       # HR / SBP         (uses BP)
    out["pulse_pressure"] = out["triage_vital_sbp"] - out["triage_vital_dbp"]      # SBP - DBP        (uses BP)
    out["spo2_rr_ratio"]  = out["triage_vital_o2"] / out["triage_vital_rr"]        # oxygen vs effort (NO BP)

    # --- added red flag combinatinns 
    out["is_tachypneic"] = (out["triage_vital_rr"]   > 20   ).astype(int)   # fast breathing
    out["is_hypoxic"]    = (out["triage_vital_o2"]   < 92   ).astype(int)   # low oxygen
    out["is_febrile"]    = (out["triage_vital_temp"] >= 100.4).astype(int)  # fever
    
    # --- a severity score = how many red flags fire:
    out["red_flag_count"] = out[["is_tachypneic", "is_hypoxic", "is_febrile"]].sum(axis=1)

    return out

X_train_fe = add_clinical_features(X_train)
X_test_fe = add_clinical_features(X_test)
print("Features after engineering:", X_train_fe.shape[1])
X_train_fe.head()

Features after engineering: 215


,triage_vital_hr,triage_vital_sbp,triage_vital_dbp,triage_vital_rr,triage_vital_o2,triage_vital_o2_device,triage_vital_temp,triage_glucose,cc_abdominalcramping,cc_abdominaldistention,...,cc_woundre-evaluation,cc_wristinjury,cc_wristpain,shock_index,pulse_pressure,spo2_rr_ratio,is_tachypneic,is_hypoxic,is_febrile,red_flag_count
35369,104.0,120.0,71.0,22.0,98.0,1.0,98.2,137.0,0.0,0.0,...,0.0,0.0,0.0,0.866667,49.0,4.454545,1,0,0,1
52043,78.0,115.0,76.0,18.0,96.0,0.0,98.4,102.0,0.0,0.0,...,0.0,0.0,0.0,0.678261,39.0,5.333333,0,0,0,0
13610,96.0,119.0,78.0,18.0,94.0,0.0,98.1,108.0,0.0,0.0,...,0.0,0.0,0.0,0.806723,41.0,5.222222,0,0,0,0
54796,89.0,128.0,93.0,16.0,98.0,0.0,97.7,108.0,0.0,0.0,...,0.0,0.0,0.0,0.695312,35.0,6.125000,0,0,0,0
11096,89.0,113.0,78.0,18.0,98.0,0.0,98.1,92.0,0.0,0.0,...,0.0,0.0,0.0,0.787611,35.0,5.444444,0,0,0,0


# M1 - Random Forest